### REGRESIÓN LINEAL EN PYTHON Y VARIABLES PROXY

##### Para esta clase utilizaremos como referencia la hipótesis y datos utilizados en el artículo: **"The Colonial Origins of Comparative Development: An Empirical Investigation"**

* Autores: Daron Acemoglu, Simon Johnson, James A. Robinson

* Fuente: The American Economic Review, Vol. 91, No. 5 (Dec., 2001), pp. 1369-1401 https://economics.mit.edu/files/4123

* La versión en español la pueden encontrar en  redalyc.org/pdf/419/41901302.pdf

#### PROPÓSITO DE ESTA CLASE
* Aplicar el concepto de variables instrumentales que ya revisamos de forma teórica.
* Utilizando un modelo de 2 etapas analizar la hipótesis de los autores: el desempeño económico observado puede ser atribuido a las diferencias institucionales.

* **Los autores del artículo proponen 3 premisas:**

* 1. Los diversos tipos de políticas de colonización crearon diferentes grupos de instituciones.  En un extremo, los europeos establecieron “Estados extractivos” , instituciones que  no proporcionaron mucha protección a la propiedad privada, ni establecieron un sistema de pesos y contrapesos contra la  expropiación del gobierno. 

* En el otro extremo, muchos europeos emigraron y se asentaron en diversas colonias, creando  “nuevas Europas”. Los colonizadores trataron de replicar las instituciones europeas, con gran énfasis en la propiedad privada y en el control del poder del gobierno.

* 2. La factibilidad de los asentamientos influyó en la estrategia de colonización. En lugares donde el ambiente insalubre no era favorable al asentamiento europeo, no resultaba posible crear “nuevas Europas”,y era más factible la formación del Estado extractivo.

* 3. El Estado colonial y las instituciones persistieron aun después de la independencia.


* **HIPÓTESIS:** las tasas de mortalidad (potenciales) de los colonizadores fueron el principal determinante de los asentamientos; los asentamientos fueron un determinante importante de las instituciones iniciales (en la práctica, las instituciones de 1900); y existe una fuerte correlación entre las instituciones iniciales y las instituciones actuales.

**¿Cómo medimos las diferencias institucionales y los resultados económicos?**

* Los resultados económicos de una economía son aproximados por el logaritmo del PIB per cápita en 1995, ajustados por la paridad del poder adquisitivo, ppp.

* Las diferencias institucionales son aproximadas mediante el  índice de protección contra la expropiación que reporta un  valor entre 0 y 10 para cada país y año,donde 0 corresponde a la menor protección contra la expropiación. Se utilizó  el valor promedio de cada país entre 1985 y 1995 ( “riesgo de expropiación” promedio 1985-95", este índice fue construido por el Grupo de Servicios de Riesgo Político https://www.prsgroup.com/).

* La principal contribución del artículo es el uso de las tasas de mortalidad como fuente de variación exógena en las diferencias institucionales.

* Dicha variación es necesaria para determinar si son las instituciones las que dan lugar a un mayor crecimiento económico, y no al revés.



### 1.- Importando las librerías necesarias
* Para esta clase necesitamos instalar linearmodels

In [ ]:
#!pip install pip
#!pip install linearmodels

In [ ]:
import numpy as np
import pandas as pd
#pd.core.common.is_list_like = pd.api.types.is_list_like
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from linearmodels.iv import IV2SLS
from linearmodels.iv.results import compare

#
import warnings
warnings.filterwarnings('ignore')

### 2.-Datos.
* Con la ayuda de la instrucción *read_stata* leeremos los datos, nuestro Dataframe contiene la siguiente información:
- shortnam - Abreviatura Nonmbre país.
- euro1900 -Restricciones al Ejecutivo en 1900
- excolony: variable dummy 1- excolonia -0 no excolonia
- logpgp95 logaritmo del pib per cápita en 1995 
- avexpr Protección promedio contra el riesgo de expropiación,1985-1995 
- cons1 Constraint on executive in first year of independence 
- cons90 Constraint on executive in 1900 
- democ00a índice de democracia 1900
- cons00a
- extmort4
- logem4 logaritmo de la tasa de mortalidad de los colonizadores para una muestra de 75 países,
- loghjypl
- baseco

In [ ]:
#
df1 = pd.read_stata('maketable1.dta')
df1.head()

### 3.- Explorando los datos de forma gráfica
* Con una gráfica de dispersión exploraremos la relación entre las variables **'avexpr' Protección promedio contra el riesgo de expropiación,1985-1995 y 'logpgp95' Logaritmo del pib per cápita, 1975 y 1995**.
* ¿Qué tipo de relación observas en los datos?

In [ ]:
# NOTA:
# 
print(plt.style.available)

In [ ]:
#
plt.style.use('seaborn-v0_8-white') #ggplot, 'seaborn-v0_8-white'
df1.plot(x ='avexpr', y ='logpgp95', kind = 'scatter')
plt.title('Gráfica 1: Gráfico de dispersión')
plt.show()

### 4.- Modelo OLS Bivariado

* Si un alto índice de  Protección promedio contra el riesgo de expropiación es una medida de calidad institucional, entonces **"mejores instituciones"  parecen estar positivamente correlacionadas con mejores desempeño económico (medido como un alto PIB per capita).**

* Para describir esta relación, podríamos generar un **modelo bivariado entre el PIB percápita como variable dependiente y el Índice de Protección  promedio contra el riesgo de expropiación  como variable explicativa.**

In [ ]:
# Generamos un subconjunto limpio de datos eliminando las observaciones con datos NA
df1_subset = df1.dropna(subset = ['logpgp95', 'avexpr'])

df1_subset.head()

In [ ]:
#
df1_subset = df1_subset[df1_subset['baseco'] == 1]

df1_subset.tail()

In [ ]:
#Establecemos los parámetros X y Y
X = df1_subset['avexpr'] 
y = df1_subset['logpgp95']

#Guardamos los valores de nuestras etiquetas 
labels = df1_subset['shortnam']

# Reemplazamos los marcadores con los nombres de las etiquetas
fig, ax = plt.subplots()
ax.scatter(X, y, marker = '')

for i, label in enumerate(labels):
    ax.annotate(label, (X.iloc[i], y.iloc[i]))

# Trazamos una línea de tendencia
ax.plot(np.unique(X),
         np.poly1d(np.polyfit(X, y, 1))(np.unique(X)),
         color = 'black')

ax.set_xlim([3.3,10.5])
ax.set_ylim([4,10.5])
ax.set_xlabel('Protección promedio contra el riesgo de expropiación 1985-95')
ax.set_ylabel('Log PIB per capita, PPP, 1995')
ax.set_title('Gráfica 2: OLS relación entre riesgo de expropiación e ingreso')

# Save the Figure
#plt.savefig("FIGURA.png", bbox_inches = 'tight')

plt.show()

* Para estimar los parámetros del modelo OLS, necesitamos añadir una columna con valor 1 

In [ ]:
#
df1['const'] = 1

df1.head()

* Ahora podemos construir el modelo OLS

In [ ]:
#
reg1 = sm.OLS( endog = df1['logpgp95'], exog = df1[['const', 'avexpr']], \
       missing = 'drop')

results = reg1.fit()

print(results.summary())

### Resultados:
**intercepto=4.63, B1=0.53,** la calidad institucional tiene un efecto positivo  en el desempeño económico.
**p-value:** El efecto de las instituciones sobre el PIB per cápita  es estadísticamente significativo
**R²=.611** alrededor de 61% de la variación en la variable LOG PIB per cápita es explicada por la variable Índice de  Protección promedio contra el riesgo de expropiación

### Utilizando nuestro modelo para predecir niveles de PIB per cápita

In [ ]:
#primero calculamos el promedio de avexpr

mean_expr = np.mean(df1_subset['avexpr'])

mean_expr

In [ ]:
results.predict(exog = [1, mean_expr] )

### Podemos obtener el valor predicho para cada valor 

In [ ]:
# Eliminamos los valores nulos de la base

df1_plot = df1.dropna( subset = ['logpgp95', 'avexpr'] )

# Graficamos los valores predichos

fig, ax = plt.subplots()
ax.scatter( df1_plot['avexpr'], 
            results.predict(), 
            alpha = 0.5,
            label = 'Predicción' )

# Graficamos los valores observados

ax.scatter( df1_plot['avexpr'], 
            df1_plot['logpgp95'], 
            alpha = 0.5, 
            label = 'Observado')

ax.legend()
ax.set_title( 'OLS valores predichos' )
ax.set_xlabel( 'avexpr' )
ax.set_ylabel( 'logpgp95' )

plt.show()

###  5.-Modelo Multivariado
* Hasta ahora, nuestro modelo sólo considera como variable explicativa el Índice de  Protección promedio contra el riesgo de expropiación, seguramente existen otras variables que tienen un efecto sobre el PIB per cápita.

* Los cálculos de los parámetros de nuestro modelo pueden estar afectados por lo que se conoce como **"sesgo de variable omitida"**, para solucionar este problema, extenderemos nuestro modelo para incluir otras variables.


In [ ]:
# cargamos los datos para nuestro modelo multivariado

df2 = pd.read_stata('maketable2.dta')

# Creamos nuestra constante
df2['const'] = 1

df2

In [ ]:
# Creamos una lista de variables que serán utilizadas en cada regresión 
# (vamos a generar 3 modelos) 
# Seleccionamos columnas mediante estas 3 listas:
X1 = ['const', 'avexpr']
X2 = ['const', 'avexpr', 'lat_abst']
X3 = ['const', 'avexpr', 'lat_abst', 'asia', 'africa', 'other']

# Estimaremos un modelo de regresión OLS por cada conjunto de variables.

reg1 = sm.OLS( df2['logpgp95'], df2[X1], missing = 'drop').fit()
reg2 = sm.OLS( df2['logpgp95'], df2[X2], missing = 'drop').fit()
reg3 = sm.OLS( df2['logpgp95'], df2[X3], missing = 'drop').fit() 

* Con la instrucción **summary_col** vamos a desplegar los resultados de los tres modelos en una sola tabla

In [ ]:
#
info_dict = { 'R-squared' : lambda x: f"{x.rsquared:.2f}",
              'No. observations' : lambda x: f"{int(x.nobs):d}" }

results_table = summary_col( results = [reg1, reg2, reg3] ,
                             float_format = '%0.2f',
                             stars = True,
                             model_names = [ 'Model 1',
                                             'Model 3',
                                             'Model 4'],
                             info_dict = info_dict,
                             regressor_order = [ 'const',
                                                 'avexpr',
                                                 'lat_abst',
                                                 'asia',
                                                 'africa' ])

results_table.add_title('Table 2 - OLS Regressions')

print(results_table)

### 6. Endogeneidad Modelo de Mínimos Cuadrados en dos etapas
* **Endogeneidad** puede surgir como resultado de un error de medición, autorregresión con autocorrelación de errores, simultaneidad y variables omitidas. Utilizando un modelo OLS de dos etapas revisaremos cómo podemos arreglar este problema.

* La relación que existe entre el Índice de protección  promedio contra el riesgo de expropiación ('avexpr') y el    Logaritmo del PIB per cápita, puede ser bidireccional. 
* Por ejemplo, es probable que los países más ricos puedan financiar o preferir mejores instituciones; o que las variables que afectan el ingreso también pueden estar correlacionadas con diferencias institucionales; también podría ser plausible que  la construcción del índice de protección  promedio contra el riesgo de expropiación pudo sesgarse,  los analistas pueden estar predispuestos a ver que los países con mayores ingresos tengan mejores instituciones

### Instrumentos y Método de Variables Intrumentales en dos etapas

* Instrumentemos nuestro índice de protección a la democracia a través de una variable instrumental: la tasa de mortalidad de los primeros colonizadores.

* De esta forma utilizaremos el procedimiento de estimación de Mínimos Cuadros en Dos Etapas. Podemos utilizar el estimador de Variables Instrumentales para determinar (Segunda Etapa):

$$\hat{\boldsymbol{\beta}}^{IV} = (\hat{\mathbf{X}}' \hat{\mathbf{X}})^{-1} \hat{\mathbf{X}}' \mathbf{Y}$$

* Por otro lado, podemoos establecer el siguiente vector de innstrumentos:
\begin{equation*}
    \mathbf{z}_i = (1, x_{i1}, \ldots, x_{iK-1}, z_{i1}, \ldots, z_{iM})
\end{equation*}

* Contruyendo de forma simimar a otras matrices a $\mathbf{Z}$ apilado la información de cada uno de los individuos. De esta forma podremos constriuir $\hat{\mathbf{X}}$ mediante el uso de un estimador de MCO:
\begin{eqnarray*}
    \hat{\mathbf{X}} & = & \mathbf{Z} \hat{\boldsymbol{\gamma}} \\
    & = & \mathbf{Z} (\mathbf{Z}' \mathbf{Z})^{-1} \mathbf{Z}' \mathbf{X}
\end{eqnarray*}

* De lo anterior tendríamos que (Primera Etapa):
\begin{equation*}
    \hat{\mathbf{X}}' = \mathbf{X}' \mathbf{Z} (\mathbf{Z}' \mathbf{Z})^{-1} \mathbf{Z}'
\end{equation*}

* Sólo para poner en contexto, podemos platear el Método Generalizado de Momentos de la siguiente forma:


$$\hat{\boldsymbol{\beta}}^{GMM} = (\hat{\mathbf{X}}' \hat{\mathbf{W}} \hat{\mathbf{X}})^{-1} \hat{\mathbf{X}}'\hat{\mathbf{W}} \mathbf{Y}$$

Donde $\hat{\mathbf{W}}$ es una matriz definida positiva.

In [ ]:
# Regresamos a nuestro DF inicial: Borrado de NA's 
df1_subset2 = df1.dropna( subset = [ 'logem4', 'avexpr' ] )

X = df1_subset2['logem4']
y = df1_subset2['avexpr']

labels = df1_subset2['shortnam']

In [ ]:
plt.style.use('seaborn-v0_8-white')

# 
fig, ax = plt.subplots()
ax.scatter(X, y, marker = '')

for i, label in enumerate(labels):
    ax.annotate(label, (X.iloc[i], y.iloc[i]))

# Línea de tendencia
ax.plot( np.unique(X),
         np.poly1d(np.polyfit(X, y, 1))(np.unique(X)),
         color = 'darkblue' )

ax.set_xlim([1.8,8.4])
ax.set_ylim([3.3,10.4])
ax.set_xlabel('Logaritmo de la mortalidad de los colonizadores')
ax.set_ylabel('Riesgo promedio de expropiación 1985-95')
ax.set_title('Figura 3: Relación entre mortalidad de los colonizadores y el riesgo de expropiación', 
             size = 14)

#
plt.show()

### Primera Etapa: Para la primera etapa requerimos instrumentar el riesgo de expropiación

$$avexpr_i = \delta_0 + \delta_1 logem4_i + \nu_i$$

In [ ]:
# Import and select the data
df4 = pd.read_stata('maketable4.dta')
df4 = df4[df4['baseco'] == 1]
df4.head()

In [ ]:
# 
df4['const'] = 1

# Regresión
results_fs = sm.OLS(df4['avexpr'],
                    df4[['const', 'logem4']],
                    missing = 'drop').fit()

print(results_fs.summary())

### Segunda Etapa: En la segunda etapa estimamos la ecuación de interés

$$logpgp95_i = β_0 + β_1 \widehat{avexpr}_i + \varepsilon_i$$

In [ ]:
# Tomamos el valor predicho de la Primera Etapa:
df4['predicted_avexpr'] = results_fs.predict()

df4.head()

In [ ]:
# Estimamos la Segunda Etapa, mediante la estimación de la ecuación de interés
results_ss = sm.OLS(df4['logpgp95'],
                    df4[['const', 'predicted_avexpr']]).fit()

print(results_ss.summary())

### Estimación en una sola instrucción: IV2SLS

In [ ]:
# Sin estimación de ajuste de errores
iv = IV2SLS( dependent = df4['logpgp95'],
             exog = df4['const'],
             endog = df4['avexpr'],
             instruments = df4[[ 'logem4' ]]).fit()#.cov_type='unadjusted')

print(iv.summary)

### Estimación de varios modelos

* Agregaremos controles al modelo, uno a uno, para ver qué tan robusta es la estimación.
* Cuidado con los argumentos de `IV2SLS`: **`endog` es sólo la variable que se instrumenta** y
`exog` son las variables exógenas, incluida la constante. Una variable de control que se declare
como `endog` instrumentada por sí misma da el mismo número, pero describe mal el modelo.

In [ ]:
# En los tres modelos la ÚNICA variable endógena es avexpr, instrumentada con logem4.
# Las demás son variables exógenas de control: van en 'exog', no en 'endog'.
iv_01 = IV2SLS( dependent = df4['logpgp95'],
                exog = df4[['const']],
                endog = df4[['avexpr']],
                instruments = df4[[ 'logem4']]).fit()#.cov_type='unadjusted')

iv_02 = IV2SLS( dependent = df4['logpgp95'],
                exog = df4[['const', 'lat_abst']],
                endog = df4[['avexpr']],
                instruments = df4[[ 'logem4']]).fit()#.cov_type='unadjusted')

iv_03 = IV2SLS( dependent = df4['logpgp95'],
                exog = df4[['const', 'lat_abst', 'africa', 'asia']],
                endog = df4[['avexpr']],
                instruments = df4[[ 'logem4']]).fit()#.cov_type='unadjusted')
#

In [ ]:
#
result = compare({'Modelo 1': iv_01, 'Modelo 2': iv_02, 'Modelo 3': iv_03})

# Imprime el resultado
print(result)

---

## 7.- Diagnóstico del instrumento: la fuerza de la primera etapa

* Hasta aquí estimamos el modelo, pero **no lo hemos sometido a ninguna prueba**. Las notas del curso
(capítulo 3, secciones *El problema de los instrumentos débiles* y *Pruebas de especificación*) plantean
tres preguntas que toda aplicación de variables instrumentales debe responder, y son las que trabajaremos
en lo que resta del cuaderno:

| Pregunta | Herramienta | ¿Qué supuesto examina? |
|---|---|---|
| ¿El instrumento es **relevante**? | $F$ de los instrumentos excluidos | Condición de rango (contrastable) |
| ¿La variable sospechosa es **endógena**? | Prueba de Durbin-Wu-Hausman | Si hace falta instrumentar |
| ¿Los instrumentos son **válidos**? | Prueba de Sargan-Hansen | Exogeneidad, sólo parcialmente |

* La **restricción de exclusión no se puede contrastar** cuando el modelo está exactamente identificado.
Ese supuesto se defiende con argumentos, no con datos: es justo lo que Acemoglu, Johnson y Robinson
hacen a lo largo del artículo.

### El estadístico $F$ de los instrumentos excluidos

* La condición de rango exige que el instrumento tenga poder explicativo sobre la variable endógena.
El diagnóstico estándar es el estadístico $F$ de la prueba conjunta de significancia de los
**instrumentos excluidos** en la primera etapa. La regla práctica de Staiger y Stock (1997) es que
$F < 10$ es señal de alarma.

* Cuidado: se trata del $F$ de los instrumentos excluidos **únicamente**, no del $F$ global de la
primera etapa, que casi siempre será grande por la presencia de las variables exógenas incluidas.
Con un solo instrumento ambos coinciden sólo si no hay controles, y además $F = t^2$.

In [ ]:
# Herramientas adicionales para esta sección
from scipy import stats
from linearmodels.iv import IVGMM

# La primera etapa ya está estimada en results_fs: avexpr = d0 + d1*logem4 + v
prueba_F = results_fs.f_test('logem4 = 0')

F_exc = float(np.squeeze(prueba_F.fvalue))
p_exc = float(np.squeeze(prueba_F.pvalue))

print('PRIMERA ETAPA:  avexpr = d0 + d1 logem4 + v')
print('-' * 52)
print(f"  Coeficiente de logem4        : {results_fs.params['logem4']:>9.4f}")
print(f"  Error estándar               : {results_fs.bse['logem4']:>9.4f}")
print(f"  Estadístico t                : {results_fs.tvalues['logem4']:>9.4f}")
print(f"  R^2                          : {results_fs.rsquared:>9.4f}")
print('-' * 52)
print(f"  F de instrumentos excluidos  : {F_exc:>9.3f}")
print(f"  Valor p                      : {p_exc:>9.2e}")
print(f"  Comprobación  t^2 = F        : {results_fs.tvalues['logem4']**2:>9.3f}")
print('-' * 52)
print('  ¿Supera el umbral de 10 de Staiger-Stock?  ->',
      'SÍ, el instrumento es fuerte' if F_exc > 10 else 'NO, instrumento débil')

In [ ]:
# El F global y el F de los instrumentos excluidos NO son lo mismo cuando hay controles.
# Repitamos la primera etapa agregando las variables exógenas del modelo 3.

fe_controles = sm.OLS( df4['avexpr'],
                       df4[['const', 'logem4', 'lat_abst', 'africa', 'asia']],
                       missing = 'drop' ).fit()

F_glob = fe_controles.fvalue
F_solo_instr = float(np.squeeze(fe_controles.f_test('logem4 = 0').fvalue))

print('PRIMERA ETAPA CON CONTROLES (lat_abst, africa, asia)')
print('-' * 52)
print(f"  Coeficiente de logem4            : {fe_controles.params['logem4']:>8.4f}")
print(f"  F GLOBAL (todos los regresores)  : {F_glob:>8.3f}   <- NO es el diagnóstico")
print(f"  F de los INSTRUMENTOS EXCLUIDOS  : {F_solo_instr:>8.3f}   <- éste es el diagnóstico")

### Lectura de los resultados

* Sin controles el instrumento es **fuerte**: $F = 22.95$, muy por encima del umbral de 10. El
coeficiente es $-0.607$, con el signo que predice la hipótesis de los autores: donde la mortalidad
de los colonizadores fue mayor, la protección contra la expropiación es hoy menor.

* Al agregar los controles, la $F$ de los instrumentos excluidos cae a $5.59$, **por debajo del
umbral**, mientras que la $F$ global se mantiene en $6.61$. Son dos números distintos y sólo el primero
diagnostica la fuerza del instrumento. Éste es el motivo por el que el modelo 3 de la tabla anterior
tiene un error estándar bastante mayor: los controles absorben buena parte de la variación del
instrumento.

* En el trabajo aplicado moderno se considera mala práctica presentar estimaciones por variables
instrumentales **sin reportar la primera etapa**. El simulador del curso
[Instrumentos débiles](https://benjov.github.io/Econometria-I-2026/sim/instrumentos.html)
permite ver, mediante simulación Monte Carlo, cómo se deforma la distribución del estimador
conforme el instrumento se debilita.

---

## 8.- Advertencia: los errores estándar de la segunda etapa "a mano" están mal

* En la sección 6 estimamos 2SLS en dos pasos con `sm.OLS` y obtuvimos **el coeficiente correcto**
($0.9443$, idéntico al de `IV2SLS`). Sin embargo, **el error estándar que imprime esa segunda etapa
es incorrecto**, y no por poco.

* La razón es que la segunda etapa manual calcula su varianza con los residuales
$y_i - \hat{\beta}_0 - \hat{\beta}_1 \widehat{avexpr}_i$, es decir, usando la variable **predicha**.
La varianza correcta del estimador de 2SLS se construye con los residuales **estructurales**,
que usan la variable **observada**:

$$ e_i = y_i - \hat{\beta}_0 - \hat{\beta}_1 \, avexpr_i $$

* Como $\widehat{avexpr}$ tiene menos variación que $avexpr$, la suma de cuadrados de la segunda etapa
manual es demasiado pequeña y el error estándar sale **subestimado**. Verifiquémoslo y reconstruyamos
a mano la fórmula correcta.

In [ ]:
N = len(df4)

# (1) Lo que imprimió la segunda etapa manual
ee_manual = results_ss.bse['predicted_avexpr']

# (2) Residuales ESTRUCTURALES: con avexpr observada, no con la predicha
e_est = ( df4['logpgp95']
          - results_ss.params['const']
          - results_ss.params['predicted_avexpr'] * df4['avexpr'] )

sigma2 = (e_est ** 2).sum() / N
X_hat  = df4[['const', 'predicted_avexpr']].values
V_clas = sigma2 * np.linalg.inv( X_hat.T @ X_hat )
ee_corregido = np.sqrt( V_clas[1, 1] )

# (3) Varianza robusta a heterocedasticidad: la forma de emparedado
Z  = df4[['const', 'logem4']].values
X  = df4[['const', 'avexpr']].values
P_z = Z @ np.linalg.inv(Z.T @ Z) @ Z.T          # matriz de proyección
X_p = P_z @ X                                    # X proyectada sobre los instrumentos
A   = np.linalg.inv( X.T @ P_z @ X )             # el "pan"
B   = ( X_p * (e_est.values ** 2)[:, None] ).T @ X_p   # la "carne"
V_rob = A @ B @ A
ee_robusto = np.sqrt( V_rob[1, 1] )

# (4) Lo que reporta linearmodels
iv_clasico = IV2SLS( dependent   = df4['logpgp95'],
                     exog        = df4['const'],
                     endog       = df4['avexpr'],
                     instruments = df4[['logem4']] ).fit( cov_type = 'unadjusted' )

iv_robusto = IV2SLS( dependent   = df4['logpgp95'],
                     exog        = df4['const'],
                     endog       = df4['avexpr'],
                     instruments = df4[['logem4']] ).fit( cov_type = 'robust' )

print('ERROR ESTÁNDAR DEL COEFICIENTE DE avexpr')
print('=' * 62)
print(f"  Segunda etapa a mano (sm.OLS)        : {ee_manual:.4f}   <- INCORRECTO")
print(f"  Corregido a mano, clásico            : {ee_corregido:.4f}")
print(f"  IV2SLS, cov_type='unadjusted'        : {iv_clasico.std_errors['avexpr']:.4f}")
print(f"  Corregido a mano, robusto            : {ee_robusto:.4f}")
print(f"  IV2SLS, cov_type='robust'            : {iv_robusto.std_errors['avexpr']:.4f}")
print('=' * 62)
print(f"  El manual subestima en un factor de  : {ee_manual / ee_corregido:.4f}")
print(f"  Coeficientes idénticos               : "
      f"{abs(results_ss.params['predicted_avexpr'] - iv_clasico.params['avexpr']):.2e}")

### Lectura de los resultados

* El coeficiente coincide hasta el último decimal, pero el error estándar de la segunda etapa manual
($0.1255$) es casi **19 % menor** que el correcto ($0.1541$). Con el error estándar robusto la diferencia
es mayor todavía ($0.1761$).

* Este es el ejemplo perfecto del tipo de error que estudiamos en la unidad 4 del curso:
**el código corre, no marca ningún error, imprime una tabla de aspecto normal y entrega un número
equivocado**. Nadie lo detecta leyendo la salida; sólo se detecta conociendo la teoría o comparando
contra una implementación de referencia.

* **Regla práctica:** la estimación en dos pasos con `sm.OLS` sirve para *entender* el método; para
*reportar* resultados hay que usar `IV2SLS`, que calcula la varianza con la fórmula
$\mathbf{V} = (\mathbf{C}'\boldsymbol{\Omega}^{-1}\mathbf{C})^{-1}\mathbf{C}'\boldsymbol{\Omega}^{-1}
\boldsymbol{\Lambda}\boldsymbol{\Omega}^{-1}\mathbf{C}(\mathbf{C}'\boldsymbol{\Omega}^{-1}\mathbf{C})^{-1}$
de las notas.

* Nótese también que `IV2SLS` usa por omisión errores **robustos**. Para comparar contra una salida
clásica hay que pedir `cov_type='unadjusted'` explícitamente, y si se desea la corrección por grados
de libertad, `debiased=True`.

---

## 9.- Primera prueba: ¿de verdad hay endogeneidad? (Durbin-Wu-Hausman)

* Instrumentar no es gratis. Si `avexpr` fuera exógena, MCO sería consistente y **más eficiente** que
2SLS, de modo que instrumentar sin necesidad cuesta precisión. La prueba de Hausman (1978) contrasta:

$$ H_0: \mathbb{E}[\mathbf{x}'_i \varepsilon_i] = 0 \quad \text{(el regresor sospechoso es exógeno)} $$

* La versión práctica es la **regresión auxiliar de Durbin-Wu-Hausman**, también llamada enfoque de
*función de control*, y consta de tres pasos:

1. Estimar la primera etapa por MCO y guardar los residuales $\hat{v}_i$.
2. Estimar por MCO la ecuación estructural **aumentada** con esos residuales:
$$ logpgp95_i = \beta_0 + \beta_1\, avexpr_i + \rho\, \hat{v}_i + error_i $$
3. Contrastar $H_0: \rho = 0$ con una prueba $t$ ordinaria.

* La intuición es directa: $\hat{v}_i$ es la parte de `avexpr` que el instrumento **no** explica. Si esa
parte ayuda a explicar el PIB una vez incluida `avexpr`, es porque está correlacionada con el error
estructural, que es la definición misma de endogeneidad.

In [ ]:
# Paso 1: residuales de la primera etapa (la parte de avexpr no explicada por el instrumento)
df4['v_hat'] = results_fs.resid

# Paso 2: ecuación estructural aumentada
dwh = sm.OLS( df4['logpgp95'],
              df4[['const', 'avexpr', 'v_hat']] ).fit()

print(dwh.summary())

In [ ]:
# Paso 3: la prueba sobre rho, y su comparación con la forma cuadrática de Hausman

# MCO sin instrumentar, para tener el otro estimador de la comparación
mco = sm.OLS( df4['logpgp95'], df4[['const', 'avexpr']] ).fit()

# Forma cuadrática de la prueba de Hausman: (b_2SLS - b_MCO)^2 / [Var(b_2SLS) - Var(b_MCO)]
dif   = iv_clasico.params['avexpr'] - mco.params['avexpr']
v_dif = iv_clasico.std_errors['avexpr'] ** 2 - mco.bse['avexpr'] ** 2
H     = dif ** 2 / v_dif
p_H   = 1 - stats.chi2.cdf(H, 1)

print('PRUEBA DE ENDOGENEIDAD')
print('=' * 64)
print(f"  MCO      : beta = {mco.params['avexpr']:.4f}   (ee {mco.bse['avexpr']:.4f})")
print(f"  2SLS     : beta = {iv_clasico.params['avexpr']:.4f}   (ee {iv_clasico.std_errors['avexpr']:.4f})")
print(f"  Diferencia                        : {dif:.4f}")
print('-' * 64)
print('  (a) Regresión auxiliar de Durbin-Wu-Hausman')
print(f"      rho                           : {dwh.params['v_hat']:.4f}")
print(f"      error estándar                : {dwh.bse['v_hat']:.4f}")
print(f"      estadístico t                 : {dwh.tvalues['v_hat']:.4f}")
print(f"      valor p                       : {dwh.pvalues['v_hat']:.6f}")
print('-' * 64)
print('  (b) Forma cuadrática de Hausman (ecuación de las notas)')
print(f"      H                             : {H:.4f}")
print(f"      valor p  (chi2 con 1 gl)      : {p_H:.6f}")
print('-' * 64)
print('  (c) Implementación de linearmodels')
print('     ', str(iv_clasico.wu_hausman()).replace('\n', '\n      '))
print('     ', str(iv_clasico.durbin()).replace('\n', '\n      '))
print('=' * 64)
print(f"  El coeficiente de avexpr en la regresión aumentada es {dwh.params['avexpr']:.4f},")
print( "  exactamente el estimador de 2SLS: la función de control reproduce 2SLS.")

### Lectura de los resultados

* Se **rechaza** la exogeneidad con holgura: $\hat{\rho} = -0.578$ con un estadístico $t = -4.92$ y
un valor p prácticamente nulo. Instrumentar está justificado.

* Las tres versiones coinciden en la conclusión pero **no dan el mismo número**, y conviene saber por qué:
la regresión auxiliar y `wu_hausman` son pruebas $F$ con corrección por grados de libertad; `durbin`
es su versión $\chi^2$ sin corrección; y la forma cuadrática de las notas usa la diferencia de matrices
de varianzas, que en muestras pequeñas es numéricamente frágil (de hecho puede no resultar definida
positiva). Todas son asintóticamente equivalentes.

* Un detalle que vale la pena notar: el coeficiente de `avexpr` en la regresión aumentada **es**
el estimador de 2SLS. Incluir $\hat{v}_i$ como control limpia la endogeneidad. Este es el
**enfoque de función de control**, que reaparecerá en el curso al estudiar modelos no lineales y
selección de muestra (capítulos 7 y 8 de las notas).

* Sustantivamente: MCO estima $0.522$ y 2SLS estima $0.944$, casi el doble. El sesgo de MCO es
**hacia abajo**, lo cual es interesante porque la historia de causalidad inversa (países ricos compran
mejores instituciones) predice lo contrario. La explicación de los autores es el **error de medición**:
el índice de protección contra la expropiación es una medida ruidosa de la calidad institucional, y el
error de medición clásico atenúa el coeficiente de MCO hacia cero. 2SLS corrige esa atenuación.

---

## 10.- Segunda prueba: ¿son válidos los instrumentos? (Sargan-Hansen)

* Esta prueba **sólo se puede aplicar si el modelo está sobreidentificado**, es decir, si hay más
instrumentos que variables endógenas ($L > K$). Con un instrumento y una variable endógena, como en
la especificación de los autores, no hay nada que contrastar: los residuales de 2SLS son ortogonales
al instrumento **por construcción**, el $R^2$ de la regresión auxiliar es cero y la prueba es vacía.
Lo comprobaremos numéricamente.

* El procedimiento de Sargan (1958) consta de tres pasos:

1. Estimar por 2SLS y obtener los residuales $e_i = y_i - \mathbf{x}_i \hat{\boldsymbol{\beta}}_{2SLS}$.
2. Regresar esos residuales, por MCO, contra **todas** las variables exógenas.
3. Calcular $S = N \cdot R^2 \sim \chi^2_{L-K}$.

* Los grados de libertad son $L - K$: el número de **restricciones de sobreidentificación**, no el
número de instrumentos. $K$ condiciones de momento se gastan en estimar los parámetros.

* Para poder aplicarla construiremos dos modelos sobreidentificados, agregando en cada caso un segundo
instrumento. Adelantamos que **los dos darán conclusiones opuestas**, lo cual es justo lo que queremos
mostrar.

In [ ]:
# Caso 0: el modelo exactamente identificado. La prueba no existe.
e_2sls = ( df4['logpgp95']
           - iv_clasico.params['const']
           - iv_clasico.params['avexpr'] * df4['avexpr'] )

aux_exacto = sm.OLS( e_2sls, df4[['const', 'logem4']] ).fit()

print('MODELO EXACTAMENTE IDENTIFICADO (L = K = 1)')
print(f"  R^2 de la regresión auxiliar : {aux_exacto.rsquared:.2e}  (cero, salvo error numérico)")
print(f"  N * R^2                      : {len(df4) * aux_exacto.rsquared:.2e}")
print( "  Los residuales de 2SLS son ortogonales al instrumento por construcción:")
print( "  no hay información contrastable. La prueba de Sargan NO APLICA.")

In [ ]:
# Dos modelos sobreidentificados. Definimos una función que hace la prueba a mano.

df4['logem4_sq'] = df4['logem4'] ** 2

def prueba_sargan( instrumentos, nombre ):
    # Estima 2SLS con los instrumentos dados y calcula la prueba de Sargan paso a paso
    modelo = IV2SLS( dependent   = df4['logpgp95'],
                     exog        = df4['const'],
                     endog       = df4['avexpr'],
                     instruments = df4[instrumentos] ).fit( cov_type = 'unadjusted' )

    # Fuerza de la primera etapa
    fe = sm.OLS( df4['avexpr'], df4[['const'] + instrumentos] ).fit()
    F  = float(np.squeeze( fe.f_test( ', '.join(f'{v} = 0' for v in instrumentos) ).fvalue ))

    # Sargan a mano
    e   = df4['logpgp95'] - modelo.params['const'] - modelo.params['avexpr'] * df4['avexpr']
    aux = sm.OLS( e, df4[['const'] + instrumentos] ).fit()
    gl  = len(instrumentos) - 1
    S   = len(df4) * aux.rsquared
    p   = 1 - stats.chi2.cdf(S, gl)

    print(f'{nombre}')
    print('-' * 66)
    print(f"  Instrumentos                  : {', '.join(instrumentos)}")
    print(f"  beta de avexpr                : {modelo.params['avexpr']:>8.4f}  (ee {modelo.std_errors['avexpr']:.4f})")
    print(f"  F de instrumentos excluidos   : {F:>8.3f}")
    print(f"  Sargan a mano  N*R^2          : {S:>8.4f}   gl = {gl}   p = {p:.4f}")
    print(f"  Sargan de linearmodels        : {modelo.sargan.stat:>8.4f}   p = {modelo.sargan.pval:.4f}")
    print(f"  Conclusión al 5 %             : {'SE RECHAZA la validez conjunta' if p < 0.05 else 'NO se rechaza'}")
    print()
    return modelo

m_lat  = prueba_sargan( ['logem4', 'lat_abst'],  'MODELO A: mortalidad y latitud como instrumentos' )
m_cuad = prueba_sargan( ['logem4', 'logem4_sq'], 'MODELO B: mortalidad y su cuadrado como instrumentos' )

In [ ]:
# Bajo heterocedasticidad, la versión correcta es la J de Hansen, que se obtiene por GMM

gmm = IVGMM( dependent   = df4['logpgp95'],
             exog        = df4['const'],
             endog       = df4['avexpr'],
             instruments = df4[['logem4', 'logem4_sq']] ).fit()

print('MODELO B, versión robusta a heterocedasticidad')
print('-' * 58)
print(f"  beta de avexpr por GMM : {gmm.params['avexpr']:.4f}  (ee {gmm.std_errors['avexpr']:.4f})")
print('  Estadística J de Hansen:')
print('   ', str(gmm.j_stat).replace('\n', '\n    '))
print()
print('  Comparación:')
print(f"    Sargan  (homocedasticidad) : {m_cuad.sargan.stat:.4f}  p = {m_cuad.sargan.pval:.4f}")
print(f"    J de Hansen (robusta)      : {gmm.j_stat.stat:.4f}  p = {gmm.j_stat.pval:.4f}")

### Lectura de los resultados

* **Modelo A** (mortalidad + latitud): $S = 0.286$ con 1 grado de libertad, $p = 0.59$. **No se rechaza.**
Los dos instrumentos producen estimaciones mutuamente consistentes ($0.944$ y $0.921$ están muy cerca).

* **Modelo B** (mortalidad + su cuadrado): $S = 6.359$, $p = 0.012$. **Se rechaza al 5 %.** La estimación
cae de $0.944$ a $0.772$ al agregar el término cuadrático. La $J$ de Hansen, robusta a
heterocedasticidad, también rechaza aunque con menor holgura ($4.498$, $p = 0.034$).

* ¿Qué significa ese rechazo? **No** que la mortalidad de los colonizadores sea un mal instrumento.
El "segundo instrumento" del modelo B no es una fuente de variación nueva, es la **no linealidad** de
la misma variable. Lo que la prueba está detectando es que la primera etapa lineal no captura bien la
relación entre mortalidad e instituciones, o que el efecto de las instituciones sobre el ingreso no es
lineal. Una prueba de sobreidentificación es, al mismo tiempo, una prueba de forma funcional, y el
rechazo no dice cuál de los dos supuestos falló.

* Las tres advertencias de las notas, que conviene tener siempre presentes:

1. La prueba contrasta si los instrumentos son **mutuamente consistentes**, no si son válidos en
términos absolutos. Si todos fueran inválidos en la misma dirección, no rechazaría.
2. El rechazo indica que **al menos uno** es inválido, pero no dice cuál. Formalmente se requiere
suponer que al menos $K$ de ellos son válidos.
3. Con instrumentos débiles la prueba tiene **poco poder**: no rechazar puede reflejar falta de
información y no validez. Con $N = 64$ observaciones, el modelo A pasa la prueba sin que eso pruebe gran cosa.

* Y una advertencia sustantiva sobre el modelo A: usar la latitud como instrumento supone que la
latitud afecta al ingreso **únicamente** a través de las instituciones. Los propios autores no lo creen,
y por eso la usan como **control** en el modelo 2, no como instrumento. El ejercicio sirve para ilustrar
la mecánica de la prueba, no para respaldar esa especificación.

---

## 11.- El estimador de Wald: forma reducida entre primera etapa

* Cuando hay un solo instrumento y una sola variable endógena, el estimador de variables instrumentales
tiene una representación muy sencilla: es el cociente entre el efecto del instrumento sobre el resultado
(la **forma reducida**) y el efecto del instrumento sobre la variable endógena (la **primera etapa**):

$$ \hat{\beta}_{IV} = \frac{ \widehat{Cov}(z, y) }{ \widehat{Cov}(z, x) }
   = \frac{ \text{coeficiente de la forma reducida} }{ \text{coeficiente de la primera etapa} } $$

* Esta es la expresión que aparece en las notas al discutir el **LATE**: el numerador es el efecto del
instrumento sobre el resultado —lo que en la literatura de evaluación se llama *efecto de intención de
tratar*— y el denominador reescala ese efecto por unidad de tratamiento efectivamente inducido.

* Verifiquemos la identidad numéricamente. Debe cumplirse hasta la precisión de la máquina.

In [ ]:
# Forma reducida: el resultado contra el instrumento, sin pasar por avexpr
forma_reducida = sm.OLS( df4['logpgp95'], df4[['const', 'logem4']] ).fit()

wald = forma_reducida.params['logem4'] / results_fs.params['logem4']

print('ESTIMADOR DE WALD')
print('=' * 64)
print(f"  Forma reducida   : logpgp95 sobre logem4 -> {forma_reducida.params['logem4']:>8.4f} "
      f"(ee {forma_reducida.bse['logem4']:.4f},  t = {forma_reducida.tvalues['logem4']:.3f})")
print(f"  Primera etapa    : avexpr   sobre logem4 -> {results_fs.params['logem4']:>8.4f} "
      f"(ee {results_fs.bse['logem4']:.4f},  t = {results_fs.tvalues['logem4']:.3f})")
print('-' * 64)
print(f"  Cociente de Wald                        : {wald:.12f}")
print(f"  Estimador de 2SLS                       : {iv_clasico.params['avexpr']:.12f}")
print(f"  Diferencia absoluta                     : {abs(wald - iv_clasico.params['avexpr']):.2e}")
print('=' * 64)
print( "  La identidad se cumple hasta la precisión de la máquina.")

### Lectura de los resultados

* La forma reducida es **directamente interpretable y no requiere ningún supuesto de exogeneidad de
`avexpr`**: un aumento de una unidad en el logaritmo de la mortalidad se asocia con un PIB per cápita
$57$ log-puntos menor. Duplicar la mortalidad, que son $\ln 2 = 0.693$ unidades, equivale a
$0.693 \times (-0.573) \approx -0.40$, es decir, alrededor de $33\%$ menos de PIB per cápita en niveles.
Es una relación fuerte y muy significativa ($t = -7.52$).

* De hecho, la significancia de la forma reducida es el mejor argumento a favor del resultado: si el
instrumento no tuviera efecto sobre el resultado, ninguna manipulación de 2SLS lo crearía. Por eso
Angrist y Pischke recomiendan **mirar siempre la forma reducida** antes que el cociente.

* Con un instrumento continuo como éste, el estimador sigue siendo un promedio ponderado de efectos
locales: identifica el efecto de las instituciones **para los países cuya calidad institucional
responde a la mortalidad colonial**, no para la población de países en general. Ésa es la lección del
LATE y es lo que hay que tener en mente al hablar de validez externa.

---

## 12.- Inferencia robusta a instrumentos débiles: Anderson-Rubin

* Cuando el instrumento es débil, el intervalo de confianza convencional
$\hat{\beta} \pm 1.96 \cdot ee$ deja de ser confiable: la distribución normal asintótica no aproxima
bien la distribución del estimador en muestras finitas y la cobertura real queda por debajo del
95 % nominal.

* La prueba de **Anderson-Rubin** evita ese problema. La idea es notar que, si $\beta_1$ fuera el valor
verdadero $\beta_1^0$, entonces $y_i - \beta_1^0 x_i$ sería el error estructural, y por exogeneidad
el instrumento no debería explicarlo. El procedimiento es:

1. Fijar un valor candidato $\beta_1^0$ y construir $y_i - \beta_1^0 \, avexpr_i$.
2. Regresar esa variable contra la constante y el instrumento.
3. Contrastar que el coeficiente del instrumento es cero.

* El **conjunto de confianza** al 95 % es el conjunto de valores $\beta_1^0$ que **no se rechazan**.
Su cobertura es correcta sin importar la fuerza del instrumento, y a diferencia del intervalo de Wald
no tiene por qué ser simétrico alrededor del estimador puntual.

In [ ]:
def ar_valor_p( b0, robusto = True ):
    # Valor p de la prueba de Anderson-Rubin para el valor candidato b0
    y_ajustada = df4['logpgp95'] - b0 * df4['avexpr']
    m = sm.OLS( y_ajustada, df4[['const', 'logem4']] )
    m = m.fit( cov_type = 'HC1' ) if robusto else m.fit()
    return float(np.squeeze( m.f_test('logem4 = 0').pvalue ))

malla   = np.arange(0.0, 3.0 + 1e-9, 0.001)
p_rob   = np.array([ ar_valor_p(b, robusto = True  ) for b in malla ])
p_clas  = np.array([ ar_valor_p(b, robusto = False ) for b in malla ])

conj_rob  = malla[ p_rob  > 0.05 ]
conj_clas = malla[ p_clas > 0.05 ]
ic_wald   = iv_robusto.conf_int().loc['avexpr']

print('CONJUNTOS DE CONFIANZA AL 95 % PARA EL EFECTO DE LAS INSTITUCIONES')
print('=' * 68)
print(f"  Wald robusto (IV2SLS)          : [{ic_wald['lower']:.3f}, {ic_wald['upper']:.3f}]")
print(f"  Anderson-Rubin, clásico        : [{conj_clas.min():.3f}, {conj_clas.max():.3f}]")
print(f"  Anderson-Rubin, robusto (HC1)  : [{conj_rob.min():.3f}, {conj_rob.max():.3f}]")
print('=' * 68)
print(f"  ¿El conjunto AR es un intervalo contiguo?  "
      f"{'sí' if np.allclose(np.diff(conj_rob), 0.001) else 'no: es la unión de varios trozos'}")
print(f"  ¿Contiene al cero?  {'sí' if conj_rob.min() <= 0 else 'no'}   "
      f"(valor p de AR en beta = 0: {ar_valor_p(0.0):.2e})")

In [ ]:
# La curva de valores p de Anderson-Rubin

fig, ax = plt.subplots( figsize = (9, 5) )

ax.plot( malla, p_rob,  color = '#2a78d6', lw = 2,   label = 'Anderson-Rubin (HC1)' )
ax.plot( malla, p_clas, color = '#4a3aa7', lw = 1.5, ls = '--', label = 'Anderson-Rubin (clásico)' )
ax.axhline( 0.05, color = 'gray', lw = 1, ls = ':' )

# Conjunto AR robusto y el intervalo de Wald
ax.axvspan( conj_rob.min(), conj_rob.max(), color = '#2a78d6', alpha = 0.10 )
ax.hlines( 0.55, ic_wald['lower'], ic_wald['upper'], color = '#eb6834', lw = 3 )
ax.plot( iv_robusto.params['avexpr'], 0.55, marker = 'o', color = '#eb6834' )

ax.annotate( 'IC de Wald al 95 %', xy = (iv_robusto.params['avexpr'], 0.55),
             xytext = (0.15, 0.62), color = '#eb6834' )
ax.annotate( 'nivel de 5 %', xy = (2.55, 0.05), xytext = (2.35, 0.10), color = 'gray' )

ax.set_xlabel( r'valor candidato $\beta_1^0$ del efecto de avexpr' )
ax.set_ylabel( 'valor p de Anderson-Rubin' )
ax.set_title( 'Conjunto de confianza de Anderson-Rubin frente al intervalo de Wald', size = 13 )
ax.set_xlim( 0, 3 )
ax.set_ylim( 0, 1 )
ax.legend( frameon = False, loc = 'upper right' )
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

### Lectura de los resultados

* Con este instrumento, que es fuerte, los dos enfoques cuentan la misma historia: el efecto es
positivo, grande y claramente distinto de cero. Pero **no coinciden**. El conjunto de Anderson-Rubin
robusto, $[0.688,\ 1.605]$, es más ancho que el intervalo de Wald, $[0.599,\ 1.289]$, y sobre todo
**está desplazado hacia la derecha**: se extiende mucho más por arriba que por abajo.

* Esa asimetría es informativa. El intervalo de Wald es simétrico por construcción, porque supone
normalidad; el de Anderson-Rubin refleja la forma real de la superficie de verosimilitud, que en un
cociente como el de Wald siempre tiene una cola larga del lado en que el denominador se acerca a cero.
Con un instrumento débil esa cola se vuelve infinita y el conjunto AR puede ser toda la recta real o
la unión de dos intervalos disjuntos, algo que el intervalo de Wald nunca podría expresar.

* Una advertencia sobre la librería: `linearmodels` tiene una propiedad llamada `anderson_rubin` en
los resultados de `IV2SLS`, pero **no es esta prueba**: es una prueba de restricciones de
sobreidentificación, hermana de la de Sargan. Es un caso de nombres que coinciden y procedimientos
que no. Hay que leer la documentación antes de reportar un número por el nombre que lleva.

---

## 13.- Ejercicios

1. **Fuerza del instrumento y controles.** Estime la primera etapa agregando los controles de uno en
uno (`lat_abst`, luego `africa`, luego `asia`) y grafique cómo cambia la $F$ de los instrumentos
excluidos. ¿A partir de qué especificación cruza el umbral de 10 hacia abajo? Relacione el resultado
con los errores estándar de los modelos 1 a 3 de la tabla comparativa.

2. **La atenuación por error de medición.** MCO estima $0.522$ y 2SLS estima $0.944$. Simule un modelo
en el que $x^* $ es la variable verdadera, $y = 1.0 \cdot x^* + u$ y usted sólo observa
$x = x^* + \eta$ con $\eta$ ruido clásico. Verifique que MCO subestima y que un instrumento válido
recupera el $1.0$. ¿Qué magnitud de ruido reproduce una brecha como la del ejercicio empírico?

3. **Sobreidentificación.** Repita la prueba de Sargan usando `africa` y `asia` como instrumentos
adicionales de `avexpr`. Reporte el estadístico, sus grados de libertad y su conclusión, y discuta
si le parece defendible la restricción de exclusión que ese ejercicio supone.

4. **La prueba de Hausman con errores robustos.** La forma cuadrática de la prueba de Hausman **deja
de ser válida** cuando se usan errores estándar robustos, porque el argumento de eficiencia que
justifica restar las varianzas se cae. Calcule la forma cuadrática con las varianzas robustas de MCO y
2SLS y compruebe qué ocurre con el denominador. ¿Qué debe hacerse en ese caso? *(Pista: la regresión
auxiliar de Durbin-Wu-Hausman sí admite errores robustos.)*

5. **Anderson-Rubin con un instrumento débil.** Tome la especificación con los tres controles, donde
la $F$ de los instrumentos excluidos cae a $5.59$, y construya para ella el conjunto de confianza de
Anderson-Rubin sobre una malla amplia, por ejemplo de $-5$ a $10$. Compare su longitud con la del
intervalo de Wald correspondiente. ¿Sigue siendo un intervalo contiguo?

6. **El estimador de Wald con un instrumento binario.** Construya una variable indicadora
`alta_mortalidad` igual a 1 si `logem4` está por encima de su mediana, úsela como único instrumento y
calcule el estimador de Wald como cociente de diferencias de medias entre los dos grupos. Verifique
que coincide con 2SLS. En términos de la ecuación del LATE de las notas, ¿quiénes son los
*cumplidores* en este ejercicio?

7. **Réplica y verificación.** El cuadro 4 de Acemoglu, Johnson y Robinson (2001) reporta un
coeficiente de $0.94$ para la muestra base. Localice en el artículo, incluido en esta carpeta, la
columna que corresponde a la estimación del modelo 1 de este cuaderno y verifique el error estándar
que reportan los autores contra el que obtuvimos con `cov_type='robust'` y con `cov_type='unadjusted'`.
¿Cuál de los dos usaron?